# ML Challenge: Customer Outcome Prediction

## Ziel
Entwickeln Sie ein Klassifikationsmodell zur Vorhersage von `Target`.

### Aufgaben
1. Daten explorieren
2. Missing Values behandeln
3. Kategoriale Variablen kodieren
4. Mindestens 3 Modelle trainieren (nutzen auch Cross Validation) und base version erstellen
5. Accuracy, Precision, Recall und F1 vergleichen
6. Hyperparameter Tuning machen und GridCV
7. Accuracy, Precision, Recall und F1 vergleichen
8. Ergebnisse interpretieren


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn import tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC


In [2]:
# Pfad anpassen, falls notwendig
df = pd.read_csv('kunden.csv')
df.head()


,PersonID,Target,ServiceLevel,FullName,Gender,Age,RelativesOnboard,Dependents,BookingID,PricePaid,Room,DepartureLocation
0,1,0,Basic,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,A
1,2,1,Premium,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,B
2,3,1,Basic,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,A
3,4,1,Premium,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,A
4,5,0,Basic,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,A


## 1. Exploration

Beim Explorieren haben wir festegestellt, dass es einige Features gibt die wir umwandeln oder sogar entfernen müssen. PersonID, Room, BookingID und Fullname werden wir entfernen weil es hier entweder viele fehlende Werte gibt oder die Werte sehr unterschiedlich sind, sodass keine Muster erlernt werden können.

## 2. Features und Target definieren

In [3]:
df["Age"] = df["Age"].fillna(
    df.groupby("Gender")["Age"].transform("median")
)

In [4]:
df["Age"].isnull().sum()

np.int64(0)

In [5]:
df["DepartureLocation"] = df["DepartureLocation"].fillna(
    df["DepartureLocation"].mode()[0]
)

In [6]:
df["DepartureLocation"].isnull().sum()

np.int64(0)

In [7]:
ids = df["PersonID"]

In [8]:
df = df.drop(["PersonID", "FullName", "BookingID", "Room"], axis=1)

In [9]:
df.head()

,Target,ServiceLevel,Gender,Age,RelativesOnboard,Dependents,PricePaid,DepartureLocation
0,0,Basic,male,22.0,1,0,7.2500,A
1,1,Premium,female,38.0,1,0,71.2833,B
2,1,Basic,female,26.0,0,0,7.9250,A
3,1,Premium,female,35.0,1,0,53.1000,A
4,0,Basic,male,35.0,0,0,8.0500,A


1. fehlende Werte aus Age anhand des median der Geschlechter
2. fehlende Werte für DepartureLocation aufgefüllt mit mode
3. Irrelevante Spalten gelöscht (PersonID, FullName, BookingID, Room) weil entweder viele leere Werte oder Werte ohne aussagekraft

## 3. Preprocessing definieren

In [10]:
df["RelativesBool"] = df["RelativesOnboard"] >= 1


In [11]:
df.head()

,Target,ServiceLevel,Gender,Age,RelativesOnboard,Dependents,PricePaid,DepartureLocation,RelativesBool
0,0,Basic,male,22.0,1,0,7.2500,A,True
1,1,Premium,female,38.0,1,0,71.2833,B,True
2,1,Basic,female,26.0,0,0,7.9250,A,False
3,1,Premium,female,35.0,1,0,53.1000,A,True
4,0,Basic,male,35.0,0,0,8.0500,A,False


In [12]:
df["OneDependents"] = df["Dependents"] == 1
df["TwoOrMoreDependents"] = df["Dependents"] > 1

In [13]:
df.head()

,Target,ServiceLevel,Gender,Age,RelativesOnboard,Dependents,PricePaid,DepartureLocation,RelativesBool,OneDependents,TwoOrMoreDependents
0,0,Basic,male,22.0,1,0,7.2500,A,True,False,False
1,1,Premium,female,38.0,1,0,71.2833,B,True,False,False
2,1,Basic,female,26.0,0,0,7.9250,A,False,False,False
3,1,Premium,female,35.0,1,0,53.1000,A,True,False,False
4,0,Basic,male,35.0,0,0,8.0500,A,False,False,False


In [14]:
df = df.drop(["RelativesOnboard", "Dependents"], axis=1)

In [15]:
df = pd.get_dummies(df, columns=["ServiceLevel", "Gender", "DepartureLocation"], drop_first=True)

In [16]:
df.head()

,Target,Age,PricePaid,RelativesBool,OneDependents,TwoOrMoreDependents,ServiceLevel_Premium,ServiceLevel_Standard,Gender_male,DepartureLocation_B,DepartureLocation_C
0,0,22.0,7.2500,True,False,False,False,False,True,False,False
1,1,38.0,71.2833,True,False,False,True,False,False,True,False
2,1,26.0,7.9250,False,False,False,False,False,False,False,False
3,1,35.0,53.1000,True,False,False,True,False,False,False,False
4,0,35.0,8.0500,False,False,False,False,False,True,False,False


In [17]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df[["Age", "PricePaid"]] = scaler.fit_transform(
    df[["Age", "PricePaid"]]
)

In [18]:
df.head()

,Target,Age,PricePaid,RelativesBool,OneDependents,TwoOrMoreDependents,ServiceLevel_Premium,ServiceLevel_Standard,Gender_male,DepartureLocation_B,DepartureLocation_C
0,0,0.271174,0.014151,True,False,False,False,False,True,False,False
1,1,0.472229,0.139136,True,False,False,True,False,False,True,False
2,1,0.321438,0.015469,False,False,False,False,False,False,False,False
3,1,0.434531,0.103644,True,False,False,True,False,False,False,False
4,0,0.434531,0.015713,False,False,False,False,False,True,False,False


In [19]:
X = df.drop("Target", axis=1)
y = df["Target"]

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=67
)

1. neues Feature für Relatives. Get_Dummies wurde hier nicht benutzt da die genaue Anzahl wie viele Relatives es gibt nicht aussagekräftig für uns war. Daher nur true/false
2. zwei neue Features für Dependents wobei noDependents aus OneDpendents und TwoOrMoreDependents abgeleitet werden kann.
3. ServiceLevel, Gender, DepartureLocation mit Get_Dummies in Kategorien gespalten
4. Age und PricePaid wurden Normalisiert um zu verhindern dass das Modell falsche Muster aus den größen der Werte lernt
5. Target entfernen damit kein Data Leak in Trainingsdaten.
6. Daten in 80% Training und 20% Test aufgeteilt

## 4. Modelle trainieren (und verbessern)

In [21]:
model = tree.DecisionTreeClassifier(max_depth=4)

In [22]:
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
 
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.7821229050279329
Precision: 0.7228915662650602
Recall: 0.7894736842105263
F1: 0.7547169811320755
[[80 23]
 [16 60]]
              precision    recall  f1-score   support

           0       0.83      0.78      0.80       103
           1       0.72      0.79      0.75        76

    accuracy                           0.78       179
   macro avg       0.78      0.78      0.78       179
weighted avg       0.79      0.78      0.78       179



## 5. Interpretation

In [23]:
# TODO: Interpretieren Sie die Ergebnisse.
# - Welches Modell ist am besten?
# - Wo sind mögliche Overfitting-Risiken?
# - Welche Features könnten besonders wichtig sein?
